# DART ablation, matched to the LRao ablation design (IID multi)

Same experimental frame as `RunLRaoBatch` — identical data construction
(seeded pools, train = ALL n samples, validation = fresh disjoint 10%·n draw,
same planted test sets), identical optimizer (Adam 5e-4, no weight decay, no
grad clipping), FULL batch, `torch.manual_seed` before net construction
(bit-identical inits per seed across all configs), per-run tracking of
**best-val / best-detection / final** — with two differences:

- the model is **DART** (DSM loss, σ=√ρ in whitened space, ZCA front-end);
- **3000 epochs** (vs LRao's 1000).

**Ablation axis: the ZCA eigenvalue floor** — `none` (pure 1/√λ wherever
λ>0; all n here exceed the 103 bands, so Σ̂ is full-rank) vs `published`
(max(1e-5·λmax, 100), clip-up). n ∈ [128, 256, 1024, 2048], ρ = 0.1
(knob), seeds 42–44.

Validation DSM loss is evaluated under a FIXED noise realization (forked
RNG) so the val signal is comparable across epochs and the training RNG
stream stays untouched — the paired-design property is preserved.

**Budget:** 2 floors × 4 n × 3 seeds = 24 runs ≈ **30–45 min on a T4**.
Resume-safe.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, json, time, copy
import numpy as np
import torch
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert os.path.exists('repro/data/pavia-u.mat')

In [ ]:
# ----------------- knobs -----------------
N_LIST  = [128, 256, 1024, 2048]
FLOORS  = ['none', 'published']       # ZCA eigenvalue floor
RHOS    = [0.1]                       # DSM noise (extend if wanted)
SEEDS   = [42, 43, 44]
MAX_EPOCHS = 3000
SNAP = 100
VAL_FRACTION = 0.1
OUT = 'results/dart_ablation'

# references on this n grid: published lines (log-interp) + the new LRao
# recipe (no cutoff, full batch, val-selected; RunLRaoBatch, 3 seeds)
_PN = [20, 40, 60, 100, 200, 500, 1000, 2000]
_PD = [0.287, 0.300, 0.327, 0.390, 0.402, 0.447, 0.537, 0.595]
_PL = [0.329, 0.367, 0.465, 0.501, 0.509, 0.565, 0.561, 0.553]
_LN = [128, 256, 1024, 2048]
_LV = [0.405, 0.553, 0.695, 0.760]            # none/full-batch, val-selected

def refs(ns):
    return ([float(np.interp(np.log(n), np.log(_PN), _PD)) for n in ns],
            [float(np.interp(np.log(n), np.log(_PN), _PL)) for n in ns],
            [float(np.interp(np.log(n), np.log(_LN), _LV)) for n in ns])

In [ ]:
# ----------------- protocol (identical to RunLRaoBatch) -----------------
import yaml
from tqdm.auto import tqdm
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import plant_targets, Whitening
from repro.core.models import ScoreNet, dsm_loss
from repro.core.detectors import dsm_additive

cfg = yaml.safe_load(open('repro/configs/iid_multi.yaml'))
cfg.update(dataset='repro/data/pavia-u.mat')


def build_data(seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    data, gt = load_hsi(cfg['dataset'])
    bkg, tgt = build_pools(data, gt.flatten(), cfg, 'multi')
    s = tgt.mean(axis=0).astype(np.float32)
    idx = np.arange(len(bkg)); rng.shuffle(idx)
    shuf = bkg[idx]
    need = max(N_LIST) + max(1, int(max(N_LIST) * VAL_FRACTION))
    assert len(shuf) >= need + int(cfg['test_size'])
    pool = shuf[:need].astype(np.float32)
    te = shuf[-int(cfg['test_size']):].astype(np.float32)
    planted, labels, _ = plant_targets(te, s, cfg['amplitude'],
                                       cfg['target_fraction'],
                                       model='additive', seed=seed)
    return pool, planted.astype(np.float32), labels, s


def zca(tr, floor):
    """ZCA whitening; floor='none' -> 1/sqrt(lam) wherever lam>0,
    floor='published' -> clip eigenvalues up to max(1e-5*lam_max, 100)."""
    X = np.asarray(tr, np.float64)
    mu = X.mean(0); Xc = X - mu
    Sig = (Xc.T @ Xc) / max(len(X) - 1, 1)
    Sig = (Sig + Sig.T) / 2
    ev, V = np.linalg.eigh(Sig)
    if floor == 'none':
        inv_sqrt = np.where(ev > 0, 1.0 / np.sqrt(np.abs(ev)), 0.0)
    else:
        f = max(float(ev[-1]) * 1e-5, 1e2)
        inv_sqrt = 1.0 / np.sqrt(np.clip(ev, f, None))
    W = V @ np.diag(inv_sqrt) @ V.T
    return Whitening(mu.astype(np.float32), W.astype(np.float32))


def metrics(labels, sc):
    return dict(pd=_pd_at_fa(labels, sc, cfg['pfa']), auc=_auc(labels, sc))

In [ ]:
# ----------------- trainer: best-val + best-detection + final -----------------
def train_one(tr, val, floor, rho, seed, label, planted, labels, s, ckpt_dir):
    torch.manual_seed(seed)                     # same init as LRao runs
    net = ScoreNet(tr.shape[1], [128], cfg['activation'],
                   whitening=zca(tr, floor)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'], weight_decay=0.0)
    Xf = torch.tensor(tr).to(DEVICE)
    Xv = torch.tensor(val).to(DEVICE)
    sigma = float(np.sqrt(rho))
    best_val, bv_state, bv_epoch = float('inf'), None, 0
    val_hist, snaps = [], []
    pbar = tqdm(range(1, MAX_EPOCHS + 1), desc=label, leave=False)
    for ep in pbar:                              # FULL batch: one step/epoch
        net.train()
        loss = dsm_loss(net, Xf, sigma)
        opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.random.fork_rng(devices=[DEVICE] if DEVICE == 'cuda' else []):
            torch.manual_seed(12345)             # fixed val noise realization
            with torch.no_grad():
                vl = float(dsm_loss(net, Xv, sigma))
        val_hist.append(round(vl, 6))
        if np.isfinite(vl) and vl < best_val:
            best_val, bv_epoch = vl, ep
            bv_state = copy.deepcopy(net.state_dict())
        if ep % SNAP == 0:
            sc = dsm_additive(planted, tr, net, s)
            snaps.append({'epoch': ep, **metrics(labels, sc)})
            pbar.set_postfix(pd=f"{snaps[-1]['pd']:.3f}", bv=bv_epoch)
    final_state = copy.deepcopy(net.state_dict())
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({'state_dict': {k: v.cpu() for k, v in bv_state.items()},
                'epoch': bv_epoch},
               os.path.join(ckpt_dir, label + '_bestval.pt'))
    torch.save({'state_dict': {k: v.cpu() for k, v in final_state.items()},
                'epoch': MAX_EPOCHS},
               os.path.join(ckpt_dir, label + '_final.pt'))
    best_snap = max(snaps, key=lambda t: t['pd'])
    out = {'bv_epoch': bv_epoch, 'bd_epoch': best_snap['epoch'],
           'bd': {'pd': best_snap['pd'], 'auc': best_snap['auc']},
           'val_hist': val_hist, 'snaps': snaps}
    for kind, state in (('bv_model', bv_state), ('final', final_state)):
        net.load_state_dict(state); net.eval()
        sc = dsm_additive(planted, tr, net, s)
        out[kind] = metrics(labels, sc)
    return out

In [ ]:
# ----------------- the sweep (resume-safe) -----------------
os.makedirs(OUT, exist_ok=True)
met_path = os.path.join(OUT, 'metrics.json')
rec = json.load(open(met_path)) if os.path.exists(met_path) else {}
rec['_meta'] = dict(n_list=N_LIST, floors=FLOORS, rhos=RHOS, seeds=SEEDS,
                    max_epochs=MAX_EPOCHS, val_fraction=VAL_FRACTION)
t0 = time.time()
for seed in SEEDS:
    pool, planted, labels, s = build_data(seed)
    for n in N_LIST:
        tr = pool[:n]
        val = pool[n:n + max(1, int(n * VAL_FRACTION))]
        for rho in RHOS:
            for fl in FLOORS:
                key = f'f{fl}_r{rho}_n{n}_s{seed}'
                if rec.get(key, {}).get('final'):
                    continue
                t1 = time.time()
                r = train_one(tr, val, fl, rho, seed, key,
                              planted, labels, s, os.path.join(OUT, 'ckpt'))
                r['sec'] = round(time.time() - t1)
                rec[key] = r
                json.dump(rec, open(met_path, 'w'))
                print(f"{key}: bv ep{r['bv_epoch']} Pd={r['bv_model']['pd']:.3f}"
                      f" | bd ep{r['bd_epoch']} Pd={r['bd']['pd']:.3f}"
                      f" | final Pd={r['final']['pd']:.3f} ({r['sec']}s)",
                      flush=True)
print(f'TOTAL {(time.time() - t0) / 3600:.2f} h')

In [ ]:
# ----------------- analysis + figures (partial-safe) -----------------
import matplotlib.pyplot as plt
from IPython.display import Image, display

FIG = os.path.join(OUT, 'figures'); os.makedirs(FIG, exist_ok=True)
rec = json.load(open(met_path))

def get(fl, rho, n, sd):
    return rec.get(f'f{fl}_r{rho}_n{n}_s{sd}')

def show(fig, name):
    fig.tight_layout()
    p = os.path.join(FIG, name + '.png')
    fig.savefig(p, dpi=200); fig.savefig(p.replace('.png', '.pdf'))
    plt.close(fig); display(Image(p, width=980))

rho = RHOS[0]
REF_DART_PUB, REF_LRAO_PUB, REF_LRAO_NEW = refs(N_LIST)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
cols = {'none': 'tab:purple', 'published': 'tab:red'}
for ax, kind, ttl in ((axes[0], 'bv_model', 'val-selected'),
                      (axes[1], 'final', 'final epoch')):
    for fl in FLOORS:
        m = [np.nanmean([r[kind]['pd'] for sd in SEEDS
                         if (r := get(fl, rho, n, sd))] or [np.nan])
             for n in N_LIST]
        sd_ = [np.nanstd([r[kind]['pd'] for sd in SEEDS
                          if (r := get(fl, rho, n, sd))] or [np.nan])
               for n in N_LIST]
        ax.errorbar(N_LIST, m, yerr=sd_, fmt='o-', color=cols[fl], lw=1.8,
                    capsize=3, label=f'DART, floor {fl}')
    ax.plot(N_LIST, REF_DART_PUB, 's--', color='darkred', lw=1.2,
            label='DART (pub, interp)')
    ax.plot(N_LIST, REF_LRAO_NEW, '^-', color='tab:green', lw=1.4,
            label='LRao new recipe (val-sel)')
    ax.plot(N_LIST, REF_LRAO_PUB, 'x-', color='k', lw=1,
            label='LRao (pub, interp)')
    ax.set_xscale('log'); ax.set_xticks(N_LIST); ax.set_xticklabels(N_LIST)
    ax.minorticks_off(); ax.grid(alpha=0.3)
    ax.set_xlabel('n'); ax.set_title(ttl)
axes[0].set_ylabel('Pd@0.1 (mean±std, 3 seeds)')
axes[1].legend(fontsize=7.5)
fig.suptitle(f'DART matched-design ablation: ZCA floor (rho={rho}, '
             f'{MAX_EPOCHS}ep, full batch)')
show(fig, 'dart_floor_matched')

# summary table
lines = ['# DART matched ablation — Pd@0.1 (mean over seeds)', '']
for kind in ('bv_model', 'bd', 'final'):
    lines += [f'## {kind}',
              '| floor | ' + ' | '.join(f'n={n}' for n in N_LIST) + ' |',
              '|' + '---|' * (len(N_LIST) + 1)]
    for fl in FLOORS:
        vals = [np.nanmean([r[kind]['pd'] for sd in SEEDS
                            if (r := get(fl, rho, n, sd))] or [np.nan])
                for n in N_LIST]
        lines.append(f'| {fl} | ' + ' | '.join(
            f'{v:.3f}' if np.isfinite(v) else '—' for v in vals) + ' |')
    lines.append('')
open(os.path.join(OUT, 'summary.md'), 'w').write('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
# ----------------- display all figures -----------------
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('results/dart_ablation/figures/*.png')):
    print(p)
    display(Image(filename=p, width=980))

In [ ]:
# ----------------- zip -----------------
!zip -qr dart_ablation_light.zip results/dart_ablation -x "*/ckpt/*"
!zip -qr dart_ablation_full.zip results/dart_ablation
!ls -lh dart_ablation_*.zip
try:
    from google.colab import files
    files.download('dart_ablation_light.zip')
except Exception as e:
    print('manual download:', e)